In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, FloatSlider, RadioButtons, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# RANDOM PROCESS PARAMETERS
# ============================================================

np.random.seed(72)

M_max = 200
N_max = 1000

# Base white-noise ensemble
W = np.random.randn(M_max, N_max + 300)

# Random phases for sinusoidal realizations
phases = np.random.uniform(0, 2 * np.pi, M_max)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_wiener_khinchin(process_type='White noise', N=500, M=80, max_lag=100, rho=0.85):

    # --------------------------------------------------------
    # CREATE ENSEMBLE
    # --------------------------------------------------------

    if process_type == 'White noise':

        X = W[:M, 300:300 + N].copy()

        process_name = 'White Noise'

    elif process_type == 'AR(1)':

        X_temp = np.zeros((M, N + 300))

        for m in range(M):

            for n in range(1, N + 300):

                X_temp[m, n] = rho * X_temp[m, n - 1] + W[m, n]

        X = X_temp[:, 300:]

        process_name = f'AR(1) Process, ρ = {rho:.2f}'

    else:

        omega0 = 0.18 * np.pi

        n = np.arange(N)

        X = np.cos(
            omega0 * n[None, :]
            + phases[:M, None]
        )

        process_name = 'Random-Phase Sinusoidal Process'

    # --------------------------------------------------------
    # REMOVE MEAN FROM EACH REALIZATION
    # --------------------------------------------------------

    X = X - np.mean(X, axis=1, keepdims=True)

    # --------------------------------------------------------
    # AUTOCORRELATION
    # --------------------------------------------------------

    max_lag = min(max_lag, N - 1)

    R_positive = np.zeros(max_lag + 1)

    for k in range(max_lag + 1):

        products = X[:, :N - k] * X[:, k:]

        R_positive[k] = np.mean(products)

    R_negative = R_positive[1:][::-1]

    R_full = np.concatenate(
        [
            R_negative,
            R_positive
        ]
    )

    lags = np.arange(
        -max_lag,
        max_lag + 1
    )

    # --------------------------------------------------------
    # PSD FROM AUTOCORRELATION
    # WIENER-KHINCHIN
    # --------------------------------------------------------

    n_fft = 2048

    R_for_fft = np.zeros(n_fft)

    R_for_fft[:max_lag + 1] = R_positive

    if max_lag > 0:

        R_for_fft[-max_lag:] = R_positive[1:][::-1]

    PSD_from_R = np.real(
        np.fft.fftshift(
            np.fft.fft(R_for_fft)
        )
    )

    PSD_from_R = np.maximum(
        PSD_from_R,
        0
    )

    # --------------------------------------------------------
    # PSD FROM ENSEMBLE-AVERAGED PERIODOGRAMS
    # --------------------------------------------------------

    periodograms = np.zeros((M, n_fft))

    for m in range(M):

        X_fft = np.fft.fftshift(
            np.fft.fft(
                X[m, :],
                n=n_fft
            )
        )

        periodograms[m, :] = (
            np.abs(X_fft) ** 2
        ) / N

    PSD_periodogram = np.mean(
        periodograms,
        axis=0
    )

    omega = np.linspace(
        -np.pi,
        np.pi,
        n_fft,
        endpoint=False
    )

    # --------------------------------------------------------
    # NORMALIZATION FOR VISUAL COMPARISON
    # --------------------------------------------------------

    if np.max(PSD_from_R) > 0:

        PSD_from_R = (
            PSD_from_R
            / np.max(PSD_from_R)
        )

    if np.max(PSD_periodogram) > 0:

        PSD_periodogram = (
            PSD_periodogram
            / np.max(PSD_periodogram)
        )

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(8.0, 6.0)
    )

    # ========================================================
    # GRAPH 1:
    # AUTOCORRELATION
    # ========================================================

    ax1.plot(
        lags,
        R_full,
        linewidth=1.8
    )

    ax1.axhline(
        0,
        color='k',
        linewidth=0.8
    )

    ax1.axvline(
        0,
        color='k',
        linestyle='--',
        linewidth=0.8
    )

    ax1.set_xlim(
        -max_lag,
        max_lag
    )

    ax1.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax1.set_ylabel(
        'Rₓ[k]',
        fontsize=11
    )

    ax1.set_title(
        f'Autocorrelation of {process_name}',
        fontsize=12,
        pad=7
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # GRAPH 2:
    # PSD
    # ========================================================

    ax2.plot(
        omega,
        PSD_from_R,
        linewidth=1.8,
        label='FFT of autocorrelation'
    )

    ax2.plot(
        omega,
        PSD_periodogram,
        linestyle='--',
        linewidth=1.5,
        label='Averaged periodogram'
    )

    ax2.set_xlim(
        -np.pi,
        np.pi
    )

    ax2.set_ylim(
        0,
        1.12
    )

    ax2.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax2.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax2.set_ylabel(
        'Normalized PSD',
        fontsize=11
    )

    ax2.set_title(
        'Wiener–Khinchin: Autocorrelation ↔ Power Spectral Density',
        fontsize=12,
        pad=7
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # --------------------------------------------------------
    # HORIZONTAL LEGEND BELOW SECOND GRAPH
    # --------------------------------------------------------

    ax2.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.24),
        ncol=2,
        fontsize=8,
        borderaxespad=0.0
    )

    # --------------------------------------------------------
    # FIGURE SPACING
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.12,
        right=0.97,
        top=0.95,
        bottom=0.16,
        hspace=0.48
    )

    plt.show()

# ============================================================
# RADIO BUTTONS
# ============================================================

process_selector = RadioButtons(
    options=[
        'White noise',
        'AR(1)',
        'Random sinusoid'
    ],
    value='White noise',
    description='Process:',
    style={'description_width': 'initial'},
    layout=Layout(width='270px')
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

N_slider = IntSlider(
    min=100,
    max=1000,
    step=50,
    value=500,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

M_slider = IntSlider(
    min=20,
    max=200,
    step=20,
    value=80,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

lag_slider = IntSlider(
    min=20,
    max=200,
    step=10,
    value=100,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

rho_slider = FloatSlider(
    min=0.0,
    max=0.98,
    step=0.02,
    value=0.85,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

N_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1000</div>'
)

M_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">200</div>'
)

lag_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">200</div>'
)

rho_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">0.98</div>'
)

# ============================================================
# ENABLE / DISABLE CONTROLS ACCORDING TO PROCESS
# ============================================================

def update_controls(change):

    if process_selector.value == 'AR(1)':

        rho_slider.disabled = False

    else:

        rho_slider.disabled = True

process_selector.observe(
    update_controls,
    names='value'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_wiener_khinchin,
    process_type=process_selector,
    N=N_slider,
    M=M_slider,
    max_lag=lag_slider,
    rho=rho_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
Wiener–Khinchin Theorem
</div>

<div style="margin-bottom:5px;">
<b>Autocorrelation:</b> Rₓ[k] measures the statistical similarity between samples separated by lag k.
</div>

<div style="margin-bottom:5px;">
<b>Power spectral density:</b> Sₓ(e<sup>jω</sup>) describes the distribution of average signal power over frequency.
</div>

<div style="margin-bottom:5px;">
<b>Wiener–Khinchin theorem:</b> for a WSS process, the PSD is the Fourier transform of the autocorrelation sequence.
</div>

<div>
<b>This notebook:</b> compares the PSD obtained from the Fourier transform of the autocorrelation with the PSD estimated by averaging periodograms over many realizations.
</div>

</div>
""")

# ============================================================
# EXTRA SPACE BELOW THEORY
# ============================================================

theory_block = VBox(
    [
        theory_html
    ],
    layout=Layout(
        margin='0px 0px 18px 0px'
    )
)

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Samples N:</div>'
)

M_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Realizations M:</div>'
)

lag_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Maximum lag:</div>'
)

rho_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">AR parameter ρ:</div>'
)

# ============================================================
# FIXED THREE-COLUMN GRID
# LABEL | SLIDER | MAXIMUM VALUE
# ============================================================

slider_grid = GridBox(
    children=[
        N_label, N_slider, N_max_label,
        M_label, M_slider, M_max_label,
        lag_label, lag_slider, lag_max_label,
        rho_label, rho_slider, rho_max_label
    ],
    layout=Layout(
        width='265px',
        grid_template_columns='110px 100px 40px',
        grid_template_rows='30px 30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# SPACE BETWEEN RADIO BUTTONS AND SLIDER GROUP
# ============================================================

slider_group = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        margin='12px 0px 0px 0px'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        process_selector,
        slider_group
    ],
    layout=Layout(
        width='285px',
        min_width='285px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px 10px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1100px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_block,
        graph_and_controls
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)